# fracture-detection — GPU training (MURA)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'fracture-detection'
import os, sys, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
pip = [sys.executable,'-m','pip','install','-q']
subprocess.run(pip+['--no-deps','-e','.'], check=True)
# Kaggle's P100 is sm_60; its bundled torch dropped sm_60 kernels -> install a build that keeps them
subprocess.run(pip+['torch==2.4.1','torchvision==0.19.1',
    '--index-url','https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run(pip+['pytorch-lightning>=2.4,<2.6','timm>=1.0.9','torchmetrics>=1.4,<1.8','grad-cam>=1.5.4','albumentations>=1.4.10','omegaconf>=2.3','mlflow>=2.14','rich>=13.7','pydicom>=2.4'], check=True)
SRC = os.path.abspath('src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')
sys.path.insert(0, SRC)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert (torch.zeros(2, device='cuda') + 1).sum().item() == 2, 'CUDA kernel smoke test failed'
import fracture; print('fracture', getattr(fracture,'__version__','ok'))


## 1 · Prepare dataset

In [ ]:
# --- locate or download MURA, build data/mura/metadata.csv from the folder tree ---
import subprocess, re, pathlib, pandas as pd
INP = pathlib.Path("/kaggle/input")
print("input:", [p.name for p in INP.iterdir()] if INP.exists() else "none")
ROOT = next((p for p in INP.rglob("MURA-v1.1") if p.is_dir()), None) if INP.exists() else None
if ROOT is None:
    print("MURA not mounted -> downloading via kaggle CLI")
    dl = pathlib.Path("/kaggle/tmp/mura"); dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","cjinny/mura-v11","-p",str(dl),"--unzip"], check=True)
    ROOT = next(p for p in dl.rglob("MURA-v1.1") if p.is_dir())
print("MURA root:", ROOT)
rows = []
for split in ("train","valid"):
    for img in (ROOT/split).rglob("*.png"):
        parts = img.relative_to(ROOT).parts   # split, XR_PART, patientN, studyK_label, imageN.png
        study_id = f"{split}/{parts[1]}/{parts[2]}/{parts[3]}"
        rows.append(dict(image_id=study_id.replace('/','__')+'__'+img.stem, study_id=study_id,
                         patient_id=parts[2], body_part=parts[1].replace('XR_','').lower(),
                         label=1 if parts[3].endswith('positive') else 0,
                         filepath=str(img), split=split))
df = pd.DataFrame(rows)
pathlib.Path("data/mura").mkdir(parents=True, exist_ok=True)
df.to_csv("data/mura/metadata.csv", index=False)
print(len(df), "images |", df.study_id.nunique(), "studies |", df.patient_id.nunique(), "patients")
print(df.groupby("body_part")["label"].agg(["count","mean"]).round(3))

## 2 · Train

In [ ]:
!python scripts/prepare_splits.py --data-dir data/mura
!python scripts/train.py --experiment strong \
    data.image_size=320 data.batch_size=24 data.num_workers=2 \
    train.max_epochs=30 train.precision=16-mixed

In [ ]:
import pathlib
ck = list(pathlib.Path('artifacts').rglob('best.ckpt')) + list(pathlib.Path('artifacts').rglob('*.json'))
assert any(pathlib.Path('artifacts').rglob('best.ckpt')) or any(pathlib.Path('artifacts').rglob('results.json')), \
    'training produced no checkpoint/results - see the log above'
print('train artifacts OK:', [str(p) for p in ck[:6]])


## 3 · Evaluate

In [ ]:
!python scripts/evaluate.py --checkpoint artifacts/best.ckpt --n-bootstrap 2000

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))